# Colombia Open — Calidad del aire

Este notebook muestra cómo cargar el dataset **`calidad_aire`** (los 6 contaminantes del Índice de Calidad del Aire — PM10, PM2.5, SO2, O3, NO2, CO — por estación de monitoreo, 2011 a la fecha, IDEAM) con `colombia-open`.

> **Por qué solo 6 de 20 "variable":** la fuente mezcla contaminantes reales con variables meteorológicas (temperatura, viento, lluvia, radiación). Solo se publican los 6 contaminantes del ICA colombiano (Resolución 2254 de 2017) — ver `docs/datasets/calidad_aire.md` para el detalle.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.12.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
aire = hub.load_polars("calidad_aire")

print(aire.shape)
aire.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `id_estacion` | string | Estación de monitoreo — clave primaria junto con `contaminante`, `tiempo_exposicion_horas` y `anio`. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Ubicación de la estación — join key hacia `municipios`. |
| `anio` | int | Año del promedio. |
| `contaminante` | string | `PM10`, `PM2.5`, `SO2`, `O3`, `NO2` o `CO`. |
| `tiempo_exposicion_horas` | int | Ventana de exposición (24h para PM10/PM2.5/SO2, 8h para O3/CO, 1h para NO2 — ventana estándar ICA). |
| `promedio` | float | Promedio anual, µg/m³. |
| `dias_excedencia` | int, nullable | Días del año que excedieron el límite normativo. |

Esquema completo: [`docs/datasets/calidad_aire.md`](../../docs/datasets/calidad_aire.md).

## PM2.5 (ventana ICA estándar: 24h) por municipio, último año disponible

In [ ]:
ultimo_anio = aire["anio"].max()

(
    aire.filter(
        (pl.col("contaminante") == "PM2.5")
        & (pl.col("tiempo_exposicion_horas") == 24)
        & (pl.col("anio") == ultimo_anio)
    )
    .group_by(["cod_mpio", "nombre_mpio"])
    .agg(pl.col("promedio").mean().round(1).alias("pm25_promedio"))
    .sort("pm25_promedio", descending=True)
    .head(10)
)

## Evolución de un contaminante en una estación

In [ ]:
estacion = aire.filter(pl.col("contaminante") == "PM10").select("id_estacion", "nombre_estacion").row(0)
print("Estación de ejemplo:", estacion)

(
    aire.filter(
        (pl.col("id_estacion") == estacion[0])
        & (pl.col("contaminante") == "PM10")
        & (pl.col("tiempo_exposicion_horas") == 24)
    )
    .sort("anio")
    .select(["anio", "promedio", "dias_excedencia", "representatividad_temporal"])
)

## Cobertura: cuántas estaciones por departamento

In [ ]:
(
    aire.select(["id_estacion", "nombre_dpto"])
    .unique()
    .group_by("nombre_dpto")
    .agg(pl.len().alias("n_estaciones"))
    .sort("n_estaciones", descending=True)
)

## Limitaciones conocidas

- **Solo 6 de 20 "variable" de la fuente** — sin meteorología (temperatura, humedad, viento, lluvia, radiación).
- **No colapsa ventanas de exposición** — filtra por la ventana estándar ICA de cada contaminante si quieres el valor "oficial" (24h para PM10/PM2.5/SO2, 8h para O3/CO, 1h para NO2).
- **Nivel de estación, no de municipio** — varios municipios grandes tienen más de una estación.
- **Cobertura geográfica desigual** — 513 estaciones, zonas rurales probablemente sin ninguna.

Documentación completa: [`docs/datasets/calidad_aire.md`](../../docs/datasets/calidad_aire.md).